# Base final: capturas x actuaciones x penas

Une las tres fuentes en una sola tabla. Guarda la base completa y una
muestra de 200 filas para revisar a mano.

Salidas: `base_final.parquet` y `muestra_revision.csv`.

In [ ]:
from pathlib import Path

import pandas as pd

DESCARGAS = Path.home() / "Downloads"

RUTA_CAPTURAS = DESCARGAS / "capturas_2006_2026.parquet"
RUTA_ACTUACIONES = DESCARGAS / "resultado_variables.parquet"
RUTA_PENAS_CSV = DESCARGAS / "Penas_delitos_ajustado.csv"
RUTA_PENAS_XLSX = DESCARGAS / "Penas_delitos_ajustado.xlsx"

RUTA_CAPTURAS_FILTRADAS = DESCARGAS / "capturas_filtradas.parquet"
RUTA_BASE_FINAL = DESCARGAS / "base_final.parquet"
RUTA_MUESTRA = DESCARGAS / "muestra_revision.csv"

COLUMNAS_CAPTURAS = [
    "IDENTIFICACION",
    "NOMBRES",
    "APELLIDOS",
    "GENERO",
    "MUNICIPIO",
    "FECHA",
    "CONDUCTA",
    "radicado_consulta",
    "scrape_estado",
]

N_MUESTRA = 200
SEMILLA = 42

for ruta in [RUTA_CAPTURAS, RUTA_ACTUACIONES]:
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró: {ruta}")


print("Rutas verificadas.")

## 1. Capturas

In [ ]:
capturas = pd.read_parquet(RUTA_CAPTURAS)

print(f"Filas cargadas: {len(capturas):,}")

faltantes = [c for c in COLUMNAS_CAPTURAS if c not in capturas.columns]
if faltantes:
    raise KeyError(f"Faltan columnas en capturas: {faltantes}")


capturas = capturas[COLUMNAS_CAPTURAS].copy()

print("\nEstado del scrapeo:")
print(capturas["scrape_estado"].value_counts(dropna=False).to_string())

In [ ]:
antes = len(capturas)

capturas = capturas[capturas["scrape_estado"] != "ERROR"].copy()

print(f"Capturas sin error de consulta: {len(capturas):,} de {antes:,} "
      f"({len(capturas) / antes:.1%})")
print(f"Radicados únicos: {capturas['radicado_consulta'].nunique():,}")

repetidos = capturas["radicado_consulta"].duplicated().sum()
print(f"Filas con radicado repetido: {repetidos:,}")

capturas.to_parquet(RUTA_CAPTURAS_FILTRADAS, index=False)
print(f"\nGuardado: {RUTA_CAPTURAS_FILTRADAS}")

capturas["scrape_estado"] = capturas["scrape_estado"].fillna("Sin radicado")

capturas.head()

## 2. Actuaciones

Vienen en formato largo, una fila por radicado y delito, asi que el merge
puede multiplicar filas de capturas.

In [ ]:
actuaciones = pd.read_parquet(RUTA_ACTUACIONES)

print(f"Filas: {len(actuaciones):,}")
print(f"Radicados únicos: {actuaciones['radicado'].nunique():,}")

filas_por_radicado = actuaciones.groupby("radicado").size()
print(f"Radicados con más de un delito: {(filas_por_radicado > 1).sum():,}")

print("\nColumnas:")
print(actuaciones.columns.tolist())

## 3. Merge capturas x actuaciones

In [ ]:
antes = len(capturas)

data = capturas.merge(
    actuaciones,
    left_on="radicado_consulta",
    right_on="radicado",
    how="left",
    indicator=True,
)

sin_actuaciones = (data["_merge"] == "left_only").sum()

print(f"Filas: {antes:,} capturas -> {len(data):,} filas")
print(f"  filas extra por delitos múltiples: {len(data) - antes:,}")
print(f"  capturas sin actuaciones: {sin_actuaciones:,} "
      f"({sin_actuaciones / antes:.1%})")


con_match = data["_merge"] == "both"
assert (data.loc[con_match, "radicado"] == data.loc[con_match, "radicado_consulta"]).all()

data = data.drop(columns=["_merge", "radicado_consulta"])

print(f"\nColumnas: {len(data.columns)}")

## 4. Penas

In [ ]:
if RUTA_PENAS_CSV.exists():
    penas = pd.read_csv(RUTA_PENAS_CSV, encoding="utf-8-sig")
    print(f"Penas leídas de: {RUTA_PENAS_CSV.name}")
elif RUTA_PENAS_XLSX.exists():
    penas = pd.read_excel(RUTA_PENAS_XLSX, sheet_name="penas")
    print(f"Penas leídas de: {RUTA_PENAS_XLSX.name}")
else:
    raise FileNotFoundError("No se encontró el archivo de penas (ni .csv ni .xlsx)")


penas = penas[["DelitoSentencia", "pena_promedio_meses"]].copy()

assert not penas["DelitoSentencia"].duplicated().any(), "Delito repetido en penas"

print(f"Delitos con pena: {len(penas)}")

In [ ]:
antes = len(data)

data = data.merge(penas, on="DelitoSentencia", how="left")

assert len(data) == antes, "El merge de penas duplicó filas"

sin_pena = data.loc[data["pena_promedio_meses"].isna(), "DelitoSentencia"]
inesperados = sin_pena.dropna().loc[lambda s: s != "otros"].value_counts()

print(f"Filas con pena asignada: {data['pena_promedio_meses'].notna().sum():,} "
      f"de {len(data):,}")
print(f"Filas sin pena: {sin_pena.isna().sum():,} sin delito, "
      f"{(sin_pena == 'otros').sum():,} clasificadas como 'otros'")


if len(inesperados):
    print("\nDelitos sin pena que NO deberían faltar (revisar el wording):")
    print(inesperados.to_string())
else:
    print("\nTodos los delitos del listado tienen pena.")

## 5. Base final

In [ ]:
ORDEN = [
    "IDENTIFICACION",
    "NOMBRES",
    "APELLIDOS",
    "GENERO",
    "MUNICIPIO",
    "FECHA",
    "CONDUCTA",
    "scrape_estado",
    "radicado",
    "FechaLegalizacion",
    "JuzgadoCaptura",
    "FechaImputacion",
    "FechaMedida",
    "TipoMedida",
    "FechaBoletaDetencion",
    "FechaBoletaSalida",
    "BoletaSalida",
    "FechaAcusacion",
    "FechaJuicioOral",
    "Preacuerdo",
    "FechaSentencia",
    "TieneSentencia",
    "Sentencia",
    "TipoSentencia",
    "JuzgadoSentencia",
    "DelitoSentencia",
    "pena_promedio_meses",
]

orden_final = [c for c in ORDEN if c in data.columns]
orden_final += [c for c in data.columns if c not in orden_final]

data = data[orden_final].reset_index(drop=True)

print(f"Base final: {len(data):,} filas x {len(data.columns)} columnas")
print(f"Personas: {data['IDENTIFICACION'].nunique():,}")
print(f"Radicados: {data['radicado'].nunique():,}")

print("\nCobertura de cada variable (% no nulo):")
print((data.notna().mean() * 100).round(1).to_string())

data.to_parquet(RUTA_BASE_FINAL, index=False)
print(f"\nGuardado: {RUTA_BASE_FINAL}")

data.head(20)

## 6. Muestra para revision

Muestreo estratificado por TipoSentencia y TipoMedida, con semilla fija.

In [ ]:
ESTRATOS = ["TipoSentencia", "TipoMedida"]

fraccion = N_MUESTRA / len(data)

partes = []

for _, grupo in data.groupby(ESTRATOS, dropna=False):
    n_grupo = min(len(grupo), max(1, round(len(grupo) * fraccion)))
    partes.append(grupo.sample(n=n_grupo, random_state=SEMILLA))


muestra = pd.concat(partes)

if len(muestra) > N_MUESTRA:
    muestra = muestra.sample(n=N_MUESTRA, random_state=SEMILLA)
elif len(muestra) < N_MUESTRA:
    resto = data.drop(index=muestra.index).sample(
        n=N_MUESTRA - len(muestra), random_state=SEMILLA
    )
    muestra = pd.concat([muestra, resto])


muestra = muestra.sort_values("radicado").reset_index(drop=True)

print(f"Muestra: {len(muestra)} filas, {muestra['radicado'].nunique()} radicados")

print("\nComposición por tipo de sentencia (0 sin sentencia, 1 condena, 2 absolución):")
print(muestra["TipoSentencia"].value_counts(dropna=False).sort_index().to_string())

print("\nComposición por tipo de medida (0 sin medida, 1 intramural, 2 domiciliaria):")
print(muestra["TipoMedida"].value_counts(dropna=False).sort_index().to_string())

muestra.to_csv(RUTA_MUESTRA, index=False, encoding="utf-8-sig")
print(f"\nGuardado: {RUTA_MUESTRA}")

muestra.head(20)